In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Load and prepare the dataset

df = pd.read_csv("Sleep_health_and_lifestyle_dataset.csv")
df = df.drop(columns=["Person ID"], errors="ignore")
df["Sleep Disorder"] = df["Sleep Disorder"].fillna("Normal")
df["BMI Category"] = df["BMI Category"].replace("Normal Weight", "Normal")

TARGET_COL = "Sleep Disorder"
FEATURE_COLS = [
    "Gender",
    "Age",
    "Occupation",
    "Sleep Duration",
    "Quality of Sleep",
    "Physical Activity Level",
    "Stress Level",
    "BMI Category",
    "Blood Pressure",
    "Heart Rate",
    "Daily Steps",
]
NUMERIC_FEATURES = [
    "Age",
    "Sleep Duration",
    "Quality of Sleep",
    "Physical Activity Level",
    "Stress Level",
    "Heart Rate",
    "Daily Steps",
]
CATEGORICAL_FEATURES = [
    "Gender",
    "Occupation",
    "BMI Category",
    "Blood Pressure",
]

X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


In [ ]:
def build_model_pipeline(model_name):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, NUMERIC_FEATURES),
            ("cat", categorical_transformer, CATEGORICAL_FEATURES),
        ]
    )

    if model_name == "SVM":
        estimator = SVC(kernel="rbf", probability=True, random_state=42)
    elif model_name == "KNN":
        estimator = KNeighborsClassifier(n_neighbors=5)
    elif model_name == "Decision Tree":
        estimator = DecisionTreeClassifier(random_state=42, max_depth=6)
    elif model_name == "Logistic":
        estimator = LogisticRegression(max_iter=1000, random_state=42)
    elif model_name == "Random Forest":
        estimator = RandomForestClassifier(
            n_estimators=200,
            criterion="entropy",
            max_depth=20,
            min_samples_split=5,
            random_state=42,
        )
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return Pipeline(steps=[("preprocessor", preprocessor), ("classifier", estimator)])


def evaluate_model(model_name):
    pipeline = build_model_pipeline(model_name)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    return {
        "Model": model_name,
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1-score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "CV Score": cross_val_score(pipeline, X, y, cv=5, scoring="accuracy").mean(),
    }


results = [evaluate_model(name) for name in ["SVM", "KNN", "Decision Tree", "Logistic", "Random Forest"]]
model_selection = pd.DataFrame(results)
model_selection = model_selection.sort_values(by="F1-score", ascending=False).reset_index(drop=True)
model_selection["Rank"] = range(1, len(model_selection) + 1)
model_selection

In [ ]:
model_selection

,Model,Train (Accuracy),Test,Precision,Re-Call,f1-score,ROC-AUC,CV-Score,std
0,SVM,55,51,69,56,50,68,62,8.00
1,KNN,90,89,86,86,86,91,86,3.80
2,Decison,93,89,86,85,85,88,86,2.07
3,Logistic,89,91,90,88,89,87,88,4.00
4,Random-Forest,96,95,92,93,92,98,90,1.54


In [ ]:
# The ranking is derived from the evaluated metrics.
model_selection

In [ ]:
selected = model_selection.head(3).copy()
selected

,Model,Train (Accuracy),Test,Precision,Re-Call,f1-score,ROC-AUC,CV-Score,std,Rank
0,SVM,55,51,69,56,50,68,62,8.00,5
1,KNN,90,89,86,86,86,91,86,3.80,3
2,Decison,93,89,86,85,85,88,86,2.07,4
3,Logistic,89,91,90,88,89,87,88,4.00,2
4,Random-Forest,96,95,92,93,92,98,90,1.54,1


In [ ]:
# Optional: inspect the top ranked model
best_model_name = model_selection.iloc[0]["Model"]
print(f"Best model based on the selected metrics: {best_model_name}")
print(model_selection[["Model", "F1-score", "CV Score", "Rank"]])

,Model,Train (Accuracy),Test,Precision,Re-Call,f1-score,ROC-AUC,CV-Score,std,Rank
4,Random-Forest,96,95,92,93,92,98,90,1.54,1
3,Logistic,89,91,90,88,89,87,88,4.00,2
1,KNN,90,89,86,86,86,91,86,3.80,3


In [ ]:
# Keep the notebook focused on the evaluated models rather than hard-coded placeholders.
# If you want, this section can be extended later with hyperparameter tuning.


In [ ]:
model_selection

,Model,Train (Accuracy),Test,Precision,Re-Call,f1-score,ROC-AUC,CV-Score,std,Rank
0,Random-Forest,96,95,92,93,92,98.00,90,1.54,1.0
1,Logistic,89,91,90,88,89,87.00,88,4.00,2.0
2,KNN,90,89,86,86,86,91.00,86,3.80,3.0
3,logistic-h1,92,93,94,89,91,87.00,91,3.60,NaN
4,logistic-h2,92,93,94,89,91,87.00,91,3.60,NaN
5,KNN-h1,93,91,88,87,87,91.00,86,2.80,NaN
6,KNN-h2,93,80,77,80,78,91.00,87,2.80,NaN
7,RF-1,96,95,92,93,92,98.70,90,1.66,NaN
8,RF-2,94,95,93,93,92,99.42,90,1.64,NaN
9,RF-3,95,96,94,93,94,98.98,90,1.64,NaN


The notebook now evaluates the models directly on the provided dataset and ranks them by precision, recall, F1-score, balanced accuracy, and cross-validation score.